 A DNA mass conservation mechanism underpins cellular mtDNA number regulation

**Authors:** Jagir R. Hussan, Asgeir Kobro-Flatmoen, Peter Ruoff, Stig W. Omholt

## Notebook Structure
1.  **Replication time limit of a single 7S DNA:** Calculations associated with replication time limi for single 7SDNA.
2.  **7S DNA residence calculation:** Calculations associated with 7S DNA residence times
3.  **Mutant Cybrid Steady-State Distributions:** Calculations for Mutant Cybrid Steady-State Distributions (Figure 3 Verification).

In [34]:
from IPython.display import display, Markdown
import numpy as np
from scipy.stats import betabinom
from scipy.integrate import odeint, quad
import pandas as pd

### Replication time limit of a single 7S DNA

In [35]:
# Parameters from Figure 2 and Figure 3 captions
T_H = 18.0           # Cell cycle duration (hours) [cite: 175]
N_0 = 490            # Initial nucleoid count [cite: 155]
N_final = 980        # Target doubling [cite: 178]
S_7_0 = 1500         # 7SDNA(0) [cite: 316, 383]
tau = 0.001          # [cite: 383]
phi = 0.001          # [cite: 383]
rho = 3.93           # [cite: 383]
beta1 = 0.592        # 7S DNA degradation rate [cite: 163]
MTDNA_PER_N = 5      # [cite: 155]
POLY_RATE = 37       # nt/s 
LENGTH_7S = 650      # nt [cite: 34]

# 1. Define Temporal Dynamics
def get_7SDNA_t(t):
    """Calculates cellular number of 7S DNA at time t[cite: 151]."""
    # Assuming growth until td; here we simplify to full cycle for integration
    return S_7_0 * (1 + tau * (t + phi)**rho)

def get_7SDNA_p(t):
    """Calculates 7S DNA production rate per hour."""
    term1 = tau * rho * (t + phi)**(rho - 1)
    term2 = beta1 * (1 + tau * (t + phi)**rho)
    return S_7_0 * (term1 + term2)

def get_N_nr(t):
    """Simplification of N_nr(t) following a sigmoid-like doubling[cite: 1023]."""
    # Based on Figure 2B/S7, N(t) transitions from N_0 to 2*N_0
    return N_0 + (N_final - N_0) * (t / T_H)**2 

# 2. Numerical Integration for D_prop
def d_prop_integrand(t, omega):
    """The term inside the integral of Eq. 10[cite: 301]."""
    # omega is in minutes, so we convert production rate (per hr) to per min
    # Or keep it as (rate_hr * omega_min) / (60 min/hr)
    p_rate = get_7SDNA_p(t)
    n_nr = get_N_nr(t)
    return (p_rate * omega) / (n_nr * MTDNA_PER_N)

def calculate_d_prop(omega):
    # Integral from 0 to 18 hours [cite: 301]
    result, _ = quad(d_prop_integrand, 0, T_H, args=(omega,))
    # Normalize by total cell cycle time in minutes 
    return result / (T_H * 60)

# 3. Execution and Validation
omega_range = [0.29, 0.65] # [cite: 305]
results = [calculate_d_prop(o) for o in omega_range]

# Residence time calculation
t_syn_sec = LENGTH_7S / POLY_RATE # 
t_syn_min = t_syn_sec / 60

In [36]:
display(Markdown(f"""
### Calculated 7S DNA Synthesis Time : {t_syn_sec:.2f} seconds
### Residence Time Range: {max(0, res_sec[0]):.1f} to {res_sec[1]:.1f} seconds
| Omega  | predicted D-loop proportion % |Residence time (seconds) |
|----------|-------|----------------|
| {omega_values[0]:.2f}| {results[0]*100:.1f} | {max(0,res_sec[0]):.2f} |
| {omega_values[1]:.2f}| {results[1]*100:.1f} | {res_sec[1]:.2f} |

"""))


### Calculated 7S DNA Synthesis Time : 17.57 seconds
### Residence Time Range: 0.0 to 21.4 seconds
| Omega  | predicted D-loop proportion % |Residence time (seconds) |
|----------|-------|----------------|
| 0.29| 2.8 | 0.00 |
| 0.65| 6.3 | 21.43 |



### 7S DNA residence calculation


In [37]:
# Omega range in minutes (synthesis + residence time)
omega_min_minutes = 0.29
omega_max_minutes = 0.65

# Polymerization rate (nucleotides per second)
pol_gamma_rate = 37 

# --- Biological constant (Implicit in the calculation) ---
# Standard length of human mitochondrial 7S DNA (D-loop strand)
# Typically  ~650nt. 
# (Note: 0.29 min * 60 sec/min * 37 nt/sec = ~644 nt, confirming ~650 is the basis)
len_7S_dna = 650 # nucleotides

# --- Calculation ---

# 1. Calculate time required strictly for synthesis (t = length / rate)
t_synthesis_seconds = len_7S_dna / pol_gamma_rate

# 2. Convert omega range to seconds
omega_min_seconds = omega_min_minutes * 60
omega_max_seconds = omega_max_minutes * 60

# 3. Derive residence time (residence = total time - synthesis time)
# Residence time cannot be negative, so we take max(0, val) for the lower bound
residence_time_min = max(0, omega_min_seconds - t_synthesis_seconds)
residence_time_max = omega_max_seconds - t_synthesis_seconds

In [38]:
display(Markdown(f"""
### Parameters

| Parameter | Value |
|----------|-------|
| Pol gamma rate | {pol_gamma_rate} nt/s |
| 7S DNA length | {len_7S_dna} nt |
| Synthesis time | {t_synthesis_seconds:.2f} seconds |
| Omega range (min) | [{omega_min_minutes}, {omega_max_minutes}] |
| Omega range (sec) | [{omega_min_seconds:.2f}, {omega_max_seconds:.2f}] |

### Calculated residence time

| Bound | Time (seconds) |
|------|----------------|
| Lower Bound | {residence_time_min:.2f} |
| Upper Bound | {residence_time_max:.2f} |
"""))



### Parameters

| Parameter | Value |
|----------|-------|
| Pol gamma rate | 37 nt/s |
| 7S DNA length | 650 nt |
| Synthesis time | 17.57 seconds |
| Omega range (min) | [0.29, 0.65] |
| Omega range (sec) | [17.40, 39.00] |

### Calculated residence time

| Bound | Time (seconds) |
|------|----------------|
| Lower Bound | 0.00 |
| Upper Bound | 21.43 |


### Mutant Cybrid Steady-State Distributions (Figure 3 Verification)

In [39]:
# --- 1. Mutant Cybrid Steady-State Simulation ---
def simulate_mtDNA_distribution(mtDNA_size, nuc_demand=165690, N=10000):
    """Replicates the logic from the supplementary notebook"""
    hist = []
    dNTP_rest_accum = 0
    base_num = int(nuc_demand / (mtDNA_size * 2))
    num_mtDNA = base_num
    
    for i in range(N): 
        if (num_mtDNA == (base_num + 1)) and (dNTP_rest_accum < mtDNA_size * 2):
            num_mtDNA = base_num
        
        dNTP_rest = nuc_demand - num_mtDNA * mtDNA_size * 2
        dNTP_rest_accum += dNTP_rest

        if dNTP_rest_accum < (mtDNA_size * 2):
            num_mtDNA = base_num
        else:
            num_mtDNA = base_num + 1
            dNTP_rest_accum += (nuc_demand - mtDNA_size * 2 * num_mtDNA)
            
        hist.append([num_mtDNA, dNTP_rest_accum])
    return np.array(hist)

# Run simulations
results_9049 = simulate_mtDNA_distribution(9049)
results_25323 = simulate_mtDNA_distribution(25323)

# Helper to calculate stats
def get_stats(arr):
    counts, freqs = np.unique(arr[:, 0], return_counts=True)
    stats = {}
    for i, count in enumerate(counts):
        subset = arr[arr[:, 0] == count, 1]
        stats[int(count)] = {
            'perc': (freqs[i] / len(arr)) * 100,
            'range': [int(min(subset)), int(max(subset))]
        }
    return stats

stats_9049 = get_stats(results_9049)
stats_25323 = get_stats(results_25323)

# --- 3. Display Markdown Tables ---
display(Markdown(f"""
| Genome Case | mtDNA Count | Population Ratio | $dNTP_{{rest}}$ Range |
| :--- | :--- | :--- | :--- |
| **Deleted (9,049 bp)** | 9 copies | {stats_9049[9]['perc']:.1f}% | {stats_9049[9]['range']} |
| | 10 copies | {stats_9049[10]['perc']:.1f}% | {stats_9049[10]['range']} |
| **Duplicated (25,323 bp)** | 3 copies | {stats_25323[3]['perc']:.1f}% | {stats_25323[3]['range']} |
| | 4 copies | {stats_25323[4]['perc']:.1f}% | {stats_25323[4]['range']} |
"""))


| Genome Case | mtDNA Count | Population Ratio | $dNTP_{rest}$ Range |
| :--- | :--- | :--- | :--- |
| **Deleted (9,049 bp)** | 9 copies | 81.6% | [2808, 18096] |
| | 10 copies | 18.4% | [2808, 5614] |
| **Duplicated (25,323 bp)** | 3 copies | 62.7% | [13752, 50640] |
| | 4 copies | 37.3% | [13752, 27498] |
